# Rain Prediction Autoencoder

Creating a PyTorch Autoencoder to reduce our input dataset dimensionality and visualizing the results to see if the extracted features can be helpful to potentially build a classifier or not.

In [7]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from mpl_toolkits.mplot3d import Axes3D
import os

## Data Preparation

In [8]:
df = pd.read_csv("data/weatherAUS.csv")
df = df.drop(['Date', 'Location', 'WindDir9am', 'WindGustDir', 'WindDir3pm'], axis=1)
df = df.dropna(how = 'any')
df.loc[df['RainToday'] == 'No', 'RainToday'] = 0
df.loc[df['RainToday'] == 'Yes', 'RainToday'] = 1
X, Y = df.drop('RainTomorrow', axis=1, inplace=False), df[['RainTomorrow']]
df.head()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
6049,17.9,35.2,0.0,12.0,12.3,48.0,6.0,20.0,20.0,13.0,1006.3,1004.4,2.0,5.0,26.6,33.4,0,No
6050,18.4,28.9,0.0,14.8,13.0,37.0,19.0,19.0,30.0,8.0,1012.9,1012.1,1.0,1.0,20.3,27.0,0,No
6052,19.4,37.6,0.0,10.8,10.6,46.0,30.0,15.0,42.0,22.0,1012.3,1009.2,1.0,6.0,28.7,34.9,0,No
6053,21.9,38.4,0.0,11.4,12.2,31.0,6.0,6.0,37.0,22.0,1012.7,1009.1,1.0,5.0,29.1,35.6,0,No
6054,24.2,41.0,0.0,11.2,8.4,35.0,17.0,13.0,19.0,15.0,1010.7,1007.4,1.0,6.0,33.6,37.6,0,No


## Modelling

In [9]:

# Normalizing Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Converting to PyTorch tensor
X_tensor = torch.FloatTensor(X_scaled)

# Converting string labels to numerical labels
label_encoder = LabelEncoder()
Y_numerical = label_encoder.fit_transform(Y.values.ravel())

# Defining Autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, input_size, encoding_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Linear(16, encoding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 16),
            nn.ReLU(),
            nn.Linear(16, input_size)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# Setting random seed for reproducibility
torch.manual_seed(42)

input_size = X.shape[1]  # Number of input features
encoding_dim = 3  # Desired number of output dimensions
model = Autoencoder(input_size, encoding_dim)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

# Training the autoencoder
num_epochs = 200
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X_tensor)
    loss = criterion(outputs, X_tensor)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Loss for each epoch
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

# Encoding the data using the trained autoencoder
encoded_data = model.encoder(X_tensor).detach().numpy()

Epoch [20/200], Loss: 0.8529
Epoch [40/200], Loss: 0.5907
Epoch [60/200], Loss: 0.4344
Epoch [80/200], Loss: 0.3485
Epoch [100/200], Loss: 0.3206
Epoch [120/200], Loss: 0.3091
Epoch [140/200], Loss: 0.3017
Epoch [160/200], Loss: 0.2960
Epoch [180/200], Loss: 0.2914
Epoch [200/200], Loss: 0.2876


## Reconstruction Quality

In [11]:
def compute_mse(original, reconstructed):
    return np.mean((original - reconstructed) ** 2)

reconstructed_data = model(X_tensor).detach().numpy()
autoencoder_mse = compute_mse(X_scaled, reconstructed_data)
print(f'Autoencoder Reconstruction MSE: {autoencoder_mse:.4f}')

Autoencoder Reconstruction MSE: 0.2875


In [12]:
from sklearn.decomposition import PCA

pca = PCA(n_components=encoding_dim)
pca_encoded = pca.fit_transform(X_scaled)
pca_reconstructed = pca.inverse_transform(pca_encoded)

pca_mse = compute_mse(X_scaled, pca_reconstructed)
print(f'PCA Reconstruction MSE: {pca_mse:.4f}')

PCA Reconstruction MSE: 0.3283
